# Building a RAG System with langChain and ChromaDB

### Introduction
Retrival-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models and external knowledge retrival. This notebook will walk you through building a complete RAG system using"
- Langchain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- Google Gemani: For embedding and large language model

In [69]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [70]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

# Vector Store
from langchain_community.vectorstores import Chroma

# Utility imports
import numpy as np
from typing import List

## Document Loading

In [71]:
from pathlib import Path

print(Path.cwd())

/Users/mdirfan/dev/python/agentic-ai/GenAi-Course-Udemy/ultimate-RAG/vector-store


In [72]:
from langchain_community.document_loaders import DirectoryLoader

# Load documents from directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = dir_loader.load()

print(f"Loaded: {len(documents)} documents")
print(f"First document preview:")
print(documents[0].page_content[:250]+"...")

Loaded: 2 documents
First document preview:
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. ...


In [73]:
for document in documents:
    print(document)

page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.

Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5,[39] capabilities and keywords for typing were added to the language, allowing optional static typing.[40] As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual release cycle and five-year support policy. Python 3.15.0rc1 (which defaults to UTF-8) is out in preview, and the stable release is expecte

## Document splitting

In [74]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators = ["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"Chunk examples:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 103 chunks from 2 documents
Chunk examples:
Content: Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of signifi...
Metadata: {'source': '../data/text_files/python_intro.txt'}


## Embedding models

In [75]:
HUGGINF_FACE_EMBEDDING_MODEL = os.getenv("HUGGING_FACE_EMBEDDING_MODEL")
HUGGINF_FACE_EMBEDDING_MODEL

'sentence-transformers/all-MiniLM-L6-v2'

In [76]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name=HUGGINF_FACE_EMBEDDING_MODEL
)

embedding

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8536.46it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

# Initialize the ChromaDB Vector Store and Stores the chunks in vector Representation

In [77]:
### Create a chromaDB vector store

persistent_directory = "../data/databases/ChromaDB"

## Initialize ChromaDB with open AI embeddings
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persistent_directory,
    collection_name="basic_rag_collection"
)

print(f"Vector store created with {vector_store._collection.count()} vectors")
print(f"Persisted to: {persistent_directory}")

Vector store created with 206 vectors
Persisted to: ../data/databases/ChromaDB


## Test the similarity search

In [78]:
query = "What are the types of machine learning?"

similar_docs = vector_store.similarity_search(query, k = 3)

similar_docs

[Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.[1] Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.'),
 Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.[1] Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machi

In [79]:
query = "What is python programming language, and how it is helping in AI/ML fields?"

similar_docs = vector_store.similarity_search(query, k = 3)

similar_docs

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a multi-paradigm programming language'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a multi-paradigm programming language'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.')]

## Advance Similarity search with Score

In [80]:
results_score = vector_store.similarity_search_with_score(query, k = 3)

results_score

[(Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a multi-paradigm programming language'),
  0.5552847981452942),
 (Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a multi-paradigm programming language'),
  0.5552847981452942),
 (Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'),
  0.570200502872467)]

### 2. Understanding similarity scores
The similarity score represents how closely related a document chunks is to your query. the scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = more similar (Closer in vector space)
- Score of 0 = identical vectors
- typical range: 0 to 2 (but can be higher)

Cosine similarity (if configured):

- Higher score = More similar
- range: -1 to 1 (1 being identical)

## Initialize LLM, RAG Chain, Prompt Template, Query the RAG system

In [81]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = os.getenv("GEMINI_MODEL"),
    api_key = os.getenv("GEMINI_API_KEY")
)

In [82]:
test_response = llm.invoke("What is LLM?")

test_response

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [83]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(
    os.getenv("OLLAMA_CHAT_MODEL")
)

llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [84]:
response = llm.invoke("What is large language model?")
response

AIMessage(content='This is one of the most common and most complex questions in modern technology.\n\nIn simple terms, **a Large Language Model (LLM) is a highly advanced artificial intelligence program designed to understand, generate, and interact using human language.**\n\nIt is not a traditional computer program; it is a sophisticated statistical tool that has been trained on colossal amounts of text data to recognize patterns and predict what word should come next, which allows it to generate text that sounds incredibly coherent, knowledgeable, and human.\n\nHere is a detailed breakdown of what an LLM is, how it works, and its limitations.\n\n***\n\n## 📚 1. The Core Concept: What is an LLM?\n\n### The Simple Analogy\nThink of an LLM as the ultimate version of your phone\'s autocomplete function.\n\nWhen you type "The capital of France is...", your phone might suggest "Paris." An LLM takes that idea and cranks it up to 11. It can predict not just the next word, but the next *paragr

## Modern RAG Chain

In [85]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

#### Convert vector store to retriever

In [86]:
retriever = vector_store.as_retriever(
    search_kwargs = {"k":3}
)

retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1350956d0>, search_kwargs={'k': 3})

In [87]:
## Create a prompt template
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [88]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

#### Create a document chain

In [89]:
document_chain = create_stuff_documents_chain(llm, prompt)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')
| StrOutputParser(), kwargs={},

### This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's resonse

In [90]:
### Create the final RAG chain
rag_chain = create_retrieval_chain(retriever, document_chain)

rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1350956d0>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you do

In [91]:
response = rag_chain.invoke({"input": "What is Deep Learning?"})

response

{'input': 'What is Deep Learning?',
 'context': [Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='Deep learning is a subset of machine learning, which is itself a subset of artificial intelligence.[20]'),
  Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='Deep learning is a subset of machine learning, which is itself a subset of artificial intelligence.[20]'),
  Document(metadata={'source': '../data/text_files/machine_learning-2.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.[1] Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.')],
 'answer': 'Deep learning is defin

In [92]:
response['answer']

'Deep learning is defined as a subset of machine learning, which in turn is a subset of artificial intelligence. Machine learning is a field that uses statistical algorithms to learn from data and perform tasks without explicit programming. Specifically, deep learning has advanced the use of neural networks, a class of statistical algorithms, allowing them to surpass many previous machine learning approaches in performance.'

In [93]:
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-"*150)

    # Using create_retrieval_chain approch
    result = rag_chain.invoke({"input": question})

    print(f"Answer: {result['answer']}")
    print("\nRetrieved context: ")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200]+"...")

    return result

# Test queries
test_questions = [
    "What are the three type of Machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n"+"="*150+"\n")

Question: What are the three type of Machine learning?
------------------------------------------------------------------------------------------------------------------------------------------------------
Answer: I do not know the three types of machine learning based on the provided context. The text defines machine learning as a field of artificial intelligence that uses statistical algorithms to learn from data and generalize to unseen data, but it does not list specific categories.

Retrieved context: 

--- Source 1 ---
Generalization
Characterizing the generalisation of various learning algorithms is an active topic of current research, especially for deep learning algorithms.[citation needed]

Statistics
Machine le...

--- Source 2 ---
Generalization
Characterizing the generalisation of various learning algorithms is an active topic of current research, especially for deep learning algorithms.[citation needed]

Statistics
Machine le...

--- Source 3 ---
Machine learning (ML) is 

## Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [94]:
## More flexible approch using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [95]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template(
    """Use the following context to answer the question.
    If you don't know the answer based on the context, say you don't know.
    Provide specific details from the context to support your answer.
    
    Context: {context}
    
    Question: {question}
    
    Answer:"""
)

custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\n    If you don't know the answer based on the context, say you don't know.\n    Provide specific details from the context to support your answer.\n\n    Context: {context}\n\n    Question: {question}\n\n    Answer:"), additional_kwargs={})])

In [96]:
# Format the output documents fot the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [97]:
# Build the chain
rag_chain_lcel = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | custom_prompt 
    | llm 
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1350956d0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\n    If you don't know the answer based on the context, say you don't know.\n    Provide specific details from the context to support your answer.\n\n    Context: {context}\n\n    Question: {question}\n\n    Answer:"), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')
| StrOutputParser()

In [98]:
rag_chain_lcel.invoke("What is deep learning?")

'Based on the context, deep learning is defined by its relationship to other fields and the technology it uses.\n\n**Answer:**\nDeep learning is categorized as a subset of machine learning, which itself is a subset of artificial intelligence. Furthermore, the field involves neural networks—a class of statistical algorithms—whose advances have allowed them to exceed the performance of many previous machine learning methods.\n\n**Support:**\n*   "Deep learning is a subset of machine learning, which is itself a subset of artificial intelligence.[20]"\n*   "Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance."'

In [99]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-"*150)

    # Method 1: Pass string directly (When using RunnablePassThrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200]+"...")

In [100]:
# Test LCEL chain
query_rag_lcel("What is the key concept in reinforcement learning")

Question: What is the key concept in reinforcement learning
------------------------------------------------------------------------------------------------------------------------------------------------------
Answer: I don't know. The context does not provide any information regarding the key concept in reinforcement learning.

Source Documents:

--- Source 1 ---
Tom M. Mitchell provided a widely quoted,[citation needed] more formal definition of the algorithms studied in the machine learning field: "A computer program is said to learn from experience E with r...

--- Source 2 ---
Tom M. Mitchell provided a widely quoted,[citation needed] more formal definition of the algorithms studied in the machine learning field: "A computer program is said to learn from experience E with r...

--- Source 3 ---
As a scientific endeavour, machine learning grew out of the quest for artificial intelligence (AI). In the early days of AI as an academic discipline, some researchers were interested in h

## Add new documents to existing vector store

In [110]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "../data/text_files/artificial_intelligence.txt",
    encoding="utf-8"
)

new_documents = loader.load()

new_documents

[Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='What is artificial intelligence (AI)?\nArtificial intelligence (AI) is a set of technologies that empowers computers to learn, reason, and perform a variety of advanced tasks in ways that used to require human intelligence, such as understanding language, analyzing data, and even providing helpful suggestions. It’s a transformational technology that can bring meaningful and positive change to people and societies and the world.\n\nIt encompasses many different disciplines, including computer science, data analytics and statistics, hardware and software engineering, linguistics, neuroscience, and even philosophy and psychology. \n\nAI is about teaching computers to do the amazing things our own brains can do, from understanding the world around them to learning new things and even coming up with fresh ideas. For instance, AI is used in optical character recognition (OCR) to pull text and data 

In [112]:
new_chunks = text_splitter.split_documents(new_documents)

new_chunks

[Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='What is artificial intelligence (AI)?\nArtificial intelligence (AI) is a set of technologies that empowers computers to learn, reason, and perform a variety of advanced tasks in ways that used to require human intelligence, such as understanding language, analyzing data, and even providing helpful suggestions. It’s a transformational technology that can bring meaningful and positive change to people and societies and the world.'),
 Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='It encompasses many different disciplines, including computer science, data analytics and statistics, hardware and software engineering, linguistics, neuroscience, and even philosophy and psychology.'),
 Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='AI is about teaching computers to do the amazing things our own brains can do

In [113]:
len(new_chunks)

17

In [114]:
## Add new documents to vector store
vector_store.add_documents(new_chunks)

['800f7875-a727-438a-b75a-c7f53573cbaf',
 'ca796537-4b27-416d-995b-b4ddb65cfa27',
 '5431c588-004d-46b4-9669-864f83f875a9',
 '08903f00-34f1-4445-823a-46ba169a9a25',
 '6e18c508-3598-4ba3-acdb-d01bb1a2236f',
 '27b12947-76e6-4acb-9f95-c69438eebd7d',
 'd524c9ff-d722-4893-9375-3ec220786e3a',
 'e22631e7-573a-49a0-8c63-f7b07a9c5c44',
 '18b9766d-097c-4fd8-b186-72b21f1c0fc7',
 '461fe7cc-c4e6-4d04-81e4-6ab6af37b168',
 '0eb0db96-90a5-46a2-8d94-02d3f00051e8',
 'd0bad71a-7248-467c-b3af-ef4fd6cf41d9',
 '74e1ad6d-9c71-45ea-997b-d098ebf618d4',
 '0686724e-d393-41e8-89a5-9e78dcf8b9ce',
 'b58db5b0-c8fe-460b-8758-7931abe9c9df',
 '0f91ba9c-7ff8-44bd-ba5f-0d628987d00d',
 '3c8f216a-8e3e-46c3-bba0-8c26a78f172c']

In [115]:
print(f"Total vectors now: {vector_store._collection.count()}")

Total vectors now: 223


In [117]:
## Query with the new vector store
new_question = "What is the concept of Artificial intelligence?"

result = query_rag_lcel(new_question)

result

Question: What is the concept of Artificial intelligence?
------------------------------------------------------------------------------------------------------------------------------------------------------
Answer: Artificial intelligence (AI) is defined as a set of technologies that empowers computers to learn, reason, and perform advanced tasks that traditionally required human intelligence.

**Supporting Details:**

*   **Core Function:** AI allows computers to perform tasks such as "understanding language, analyzing data, and even providing helpful suggestions." It is fundamentally about "teaching computers to do the amazing things our own brains can do," including "understanding the world around them to learning new things and even coming up with fresh ideas."
*   **Scope:** The context specifies that "AI isn't a single technology but a broad field encompassing several key areas."
*   **Mechanism:** AI systems work by relying on "data, algorithms, and computational power." They 

## Advanced RAG Technique - Conversational memory

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagePlaceholder: Placeholder for chat history in prompt
- HumanMessage/AIMessage: Structured message types for conversation history

In [119]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [120]:
## Create a prompt that includes the the chat history
contextualize_question_system_prompt = """Given a chat history and the latest user question which might reference context in the chat history,
Formulate a standalone question which can be understood without the chat history.
Do not answer the question, just reformulate it if needed and otherwise return it as is."""

contextualize_question_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_question_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

In [121]:
## Create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_question_prompt
)

history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1350956d0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMess

In [122]:
qa_system_prompt = """You are an assistent for question-answering task.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
use three sentences maximum and keep the answer consise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ('human', "{input}")
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational rag chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

In [123]:
chat_history = []

result1 = conversational_rag_chain.invoke(
    {
        "chat_history": chat_history,
        "input": "What is machine learning?"
    }
)

print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine Learning (ML) is a field of artificial intelligence that studies statistical algorithms capable of learning from data. These algorithms identify patterns and generalize knowledge to unseen data, allowing systems to perform tasks without being explicitly programmed. Advances, particularly in deep learning with neural networks, have enabled ML to surpass previous methods in performance.


In [126]:
chat_history.extend(
    [
        HumanMessage(content="What is machine learning?"),
        AIMessage(content=result1['answer'])
    ]
)

In [127]:
chat_history

[HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine Learning (ML) is a field of artificial intelligence that studies statistical algorithms capable of learning from data. These algorithms identify patterns and generalize knowledge to unseen data, allowing systems to perform tasks without being explicitly programmed. Advances, particularly in deep learning with neural networks, have enabled ML to surpass previous methods in performance.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [128]:
## Follow up question
result2 = conversational_rag_chain.invoke(
    {
        "chat_history": chat_history,
        "input": "What are it's main type?"
    }
)

result2

{'chat_history': [HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine Learning (ML) is a field of artificial intelligence that studies statistical algorithms capable of learning from data. These algorithms identify patterns and generalize knowledge to unseen data, allowing systems to perform tasks without being explicitly programmed. Advances, particularly in deep learning with neural networks, have enabled ML to surpass previous methods in performance.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': "What are it's main type?",
 'context': [Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='Machine Learning (ML): This is a type of AI where systems learn from data to identify patterns and make predictions or decisions without direct programming. Imagine teaching a computer to recognize a bird by showing it thousands of bird pi